In [2]:
import pandas as pd
import numpy as np

file_path = '../data/interim/cleaning_data/Data/quotes_merged_titles_colon.csv'

try:
    df = pd.read_csv(file_path)

    # --- Define conditions for 'unknown' values ---
    lang_unknown_mask = df['language_or_country'].isna() | \
                        (df['language_or_country'].astype(str).str.strip() == '') | \
                        (df['language_or_country'].astype(str).str.lower() == 'unknown') | \
                        (df['language_or_country'].astype(str).str.lower() == 'not found')
    
    pub_date_unknown_mask = df['publication_date'].astype(str).str.lower() == 'unknown'


    # --- Basic Statistics ---
    total_rows = len(df)
    unique_authors = df['AUTHOR'].nunique()
    unique_books = df['TITLE'].nunique()
    
    # Calculate unique authors/books with known info
    unique_authors_lang_known = df[~lang_unknown_mask]['AUTHOR'].nunique()
    unique_books_pub_date_known = df[~pub_date_unknown_mask]['TITLE'].nunique()

    print("--- Basic Dataset Statistics ---")
    print(f"Total rows: {total_rows}")
    print(f"Total unique authors: {unique_authors}")
    print(f"Unique authors with known language: {unique_authors_lang_known}")
    print(f"Total unique books: {unique_books}")
    print(f" Unique books with known publication date: {unique_books_pub_date_known}")
    

    # --- Row-level Availability Analysis ---
    both_known = (~lang_unknown_mask & ~pub_date_unknown_mask).sum()
    only_lang_known = (~lang_unknown_mask & pub_date_unknown_mask).sum()
    only_pub_date_known = (lang_unknown_mask & ~pub_date_unknown_mask).sum()
    both_unknown = (lang_unknown_mask & pub_date_unknown_mask).sum()

    print("--- Data Availability (per row) ---")
    print(f"Rows where both language and publication date are known: {both_known}")
    print(f"Rows where only language is known: {only_lang_known}")
    print(f"Rows where only publication date is known: {only_pub_date_known}")
    print(f"Rows where both language and publication date are unknown: {both_unknown}")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")


--- Basic Dataset Statistics ---
Total rows: 225228
Total unique authors: 5428
Unique authors with known language: 3678
Total unique books: 22581
 Unique books with known publication date: 9256
--- Data Availability (per row) ---
Rows where both language and publication date are known: 147949
Rows where only language is known: 60041
Rows where only publication date is known: 3200
Rows where both language and publication date are unknown: 14038


In [3]:
# Define paths and parameters for the sampling
output_path = 'random_book_sample_occurence_weighted.csv'
sample_size = 5000
random_seed = 42

try:
    # --- Step 1: Filter using the masks from the previous cell ---
    filtered_df = df[~lang_unknown_mask & ~pub_date_unknown_mask]

    # --- Step 2: Calculate weights based on book frequency ---
    book_counts = filtered_df['TITLE'].value_counts()
    
    # Check if there are enough unique books to sample from
    if len(book_counts) < sample_size:
        print(f"Error: Only found {len(book_counts)} unique books with known data, but requested a sample of {sample_size}.")
    else:
        # --- Step 3: Perform weighted random sampling for unique book titles ---
        sampled_titles = book_counts.sample(n=sample_size, weights=book_counts, replace=False, random_state=random_seed).index
        
        # --- Step 4: Get one random quote for each sampled book title ---
        sampled_df = filtered_df[filtered_df['TITLE'].isin(sampled_titles)]
        final_sample = sampled_df.groupby('TITLE', group_keys=False).sample(n=1, random_state=random_seed)

        # Shuffle the dataframe to ensure random order, as groupby sorts by title by default
        final_sample = final_sample.sample(frac=1, random_state=random_seed).reset_index(drop=True)

        # --- Step 5: Save the result to a new CSV file ---
        final_sample.to_csv(output_path, index=False)
        
        print(f"Successfully created a random sample of {len(final_sample)} unique books.")
        print(f"File saved to: '{output_path}'")

except NameError:
    print("Error: Make sure you have run the previous cell to define the 'df' and mask variables.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully created a random sample of 5000 unique books.
File saved to: 'random_book_sample_occurence_weighted.csv'


In [4]:
import pandas as pd
import numpy as np

# Define paths and parameters for the new sampling
existing_sample_path = 'random_book_sample_occurence_weighted.csv'
output_path_likes = 'random_book_sample_likes_weighted.csv'
sample_size_likes = 5000
random_seed = 42

try:
    # --- Step 1: Load the previously sampled books to exclude them ---
    existing_sample_df = pd.read_csv(existing_sample_path)
    existing_book_titles = existing_sample_df['TITLE'].unique()
    print(f"Loaded {len(existing_book_titles)} unique book titles from '{existing_sample_path}' to exclude from the new sample.")

    # --- Step 2: Prepare the dataframe for the new sampling ---
    # Filter the original dataframe using the masks from the first cell
    filtered_df = df[~lang_unknown_mask & ~pub_date_unknown_mask]
    
    # Exclude books that are already in the first sample
    available_df = filtered_df[~filtered_df['TITLE'].isin(existing_book_titles)]
    print(f"Found {available_df['TITLE'].nunique()} unique books available for the new sample.")

    # --- Step 3: Calculate weights based on 'LIKES' ---
    # Check if the 'LIKES' column exists
    if 'LIKES' not in available_df.columns:
        raise ValueError("The 'LIKES' column was not found in the DataFrame.")

    # Create a copy to avoid SettingWithCopyWarning
    sampling_df = available_df.copy()

    # Ensure the 'LIKES' column is numeric, coercing errors to NaN and then filling with 0
    sampling_df['LIKES'] = pd.to_numeric(sampling_df['LIKES'], errors='coerce').fillna(0)

    # Calculate the total likes for each book. This will be our weight.
    book_likes = sampling_df.groupby('TITLE')['LIKES'].sum()

    # We can only sample books that have more than 0 likes
    positive_likes_books = book_likes[book_likes > 0]
    
    print(f"Found {len(positive_likes_books)} books with more than 0 likes.")

    # --- Step 4: Perform weighted random sampling ---
    # Check if we have enough books for the desired sample size
    if len(positive_likes_books) < sample_size_likes:
        print(f"Warning: Only {len(positive_likes_books)} books with likes are available, which is less than the desired sample size of {sample_size_likes}.")
        current_sample_size = len(positive_likes_books)
    else:
        current_sample_size = sample_size_likes

    # Perform weighted sampling on the book titles
    sampled_titles_likes = positive_likes_books.sample(
        n=current_sample_size, 
        weights=positive_likes_books, 
        replace=False, 
        random_state=random_seed
    ).index

    # --- Step 5: Create the final sample dataframe ---
    # Get all rows for the sampled titles
    final_sample_df_likes = sampling_df[sampling_df['TITLE'].isin(sampled_titles_likes)]
    
    # Get one random quote for each sampled book title
    final_sample_likes = final_sample_df_likes.groupby('TITLE', group_keys=False).sample(n=1, random_state=random_seed)

    # Shuffle the dataframe for random order
    final_sample_likes = final_sample_likes.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    # --- Step 6: Save the new sample to a CSV file ---
    final_sample_likes.to_csv(output_path_likes, index=False)
    
    print(f"\nSuccessfully created a new sample of {len(final_sample_likes)} unique books, weighted by likes.")
    print(f"File saved to: '{output_path_likes}'")

except FileNotFoundError:
    print(f"Error: The file '{existing_sample_path}' was not found. Please make sure to run the previous cell to generate it.")
except NameError as e:
    print(f"Error: A required variable is not defined. Make sure you have run the initial cells in the notebook. Details: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Loaded 5000 unique book titles from 'random_book_sample_occurence_weighted.csv' to exclude from the new sample.
Found 4047 unique books available for the new sample.
Found 4047 books with more than 0 likes.

Successfully created a new sample of 4047 unique books, weighted by likes.
File saved to: 'random_book_sample_likes_weighted.csv'


In [5]:
import pandas as pd

# Define the paths to your two sample files
occurence_weighted_path = 'random_book_sample_occurence_weighted.csv'
likes_weighted_path = 'random_book_sample_likes_weighted.csv'

try:
    # Load the datasets
    df_occurence = pd.read_csv(occurence_weighted_path)
    df_likes = pd.read_csv(likes_weighted_path)

    # --- Calculate average likes for the occurrence-weighted sample ---
    # Ensure 'LIKES' column is numeric, coercing errors to NaN, then filling with 0
    df_occurence['LIKES'] = pd.to_numeric(df_occurence['LIKES'], errors='coerce').fillna(0)
    avg_likes_occurence = df_occurence['LIKES'].mean()

    # --- Calculate average likes for the likes-weighted sample ---
    df_likes['LIKES'] = pd.to_numeric(df_likes['LIKES'], errors='coerce').fillna(0)
    avg_likes_likes = df_likes['LIKES'].mean()

    # --- Print the comparison ---
    print("--- Average Likes Comparison ---")
    print(f"Occurrence-Weighted Sample: {avg_likes_occurence:.2f} average likes")
    print(f"Likes-Weighted Sample:      {avg_likes_likes:.2f} average likes")

except FileNotFoundError as e:
    print(f"Error: Could not find the file -> {e.filename}")
    print("Please make sure both CSV files exist in the same directory as the notebook.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


--- Average Likes Comparison ---
Occurrence-Weighted Sample: 73.86 average likes
Likes-Weighted Sample:      82.35 average likes


In [6]:
import pandas as pd

# Define the paths to your files
master_list_path = 'random_book_sample_occurence_weighted.csv'
output_log_path = 'google_books_sample_output.csv'

try:
    # Load both CSV files
    df_master = pd.read_csv(master_list_path)
    df_output = pd.read_csv(output_log_path)

    # --- Pre-computation for efficiency ---
    # Create a set of unique titles from the master list for fast lookups
    master_titles_set = set(df_master['TITLE'].unique())

    # --- Find the last matching book ---
    last_matching_title = None
    
    # Iterate through the output log in REVERSE order
    for title in reversed(df_output['TITLE'].tolist()):
        if title in master_titles_set:
            # The first one we find is the last match
            last_matching_title = title
            break # Stop searching as soon as we find it

    # --- Report the result ---
    if last_matching_title:
        # Get the original index from the master dataframe
        match_index = df_master[df_master['TITLE'] == last_matching_title].index[0]
        
        print(f"The last matching book found was: '{last_matching_title}'")
        print(f"It is at row (index): {match_index} in '{master_list_path}'")
        print(f"You should continue processing from row {match_index + 1}.")
    else:
        print("No matching books were found between the output log and the master list.")

except FileNotFoundError as e:
    print(f"Error: Could not find the file -> {e.filename}")
    print("Please make sure both CSV files exist and the paths are correct.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

The last matching book found was: 'The Black Book'
It is at row (index): 4997 in 'random_book_sample_occurence_weighted.csv'
You should continue processing from row 4998.


In [10]:
import pandas as pd
import numpy as np

# Define paths and parameters
existing_sample_path_1 = 'random_book_sample_occurence_weighted.csv'
existing_sample_path_2 = 'random_book_sample_likes_weighted.csv'
output_path_partial = 'random_book_sample_partial_data.csv'
sample_size_per_group = 1000
random_seed = 42

try:
    # --- Step 1: Load existing samples to exclude their titles ---
    df_existing_1 = pd.read_csv(existing_sample_path_1)
    titles_existing_1 = set(df_existing_1['TITLE'].unique())
    
    df_existing_2 = pd.read_csv(existing_sample_path_2)
    titles_existing_2 = set(df_existing_2['TITLE'].unique())
    
    all_excluded_titles = titles_existing_1.union(titles_existing_2)
    print(f"Loaded {len(all_excluded_titles)} unique book titles from previous samples to exclude.")

    # --- Step 2: Sample books with ONLY publication date known ---
    print("\n--- Sampling books with only publication date ---")
    
    # Filter for rows where only publication date is known
    only_date_df = df[lang_unknown_mask & ~pub_date_unknown_mask]
    
    # Exclude books from previous samples
    available_only_date_df = only_date_df[~only_date_df['TITLE'].isin(all_excluded_titles)]
    unique_titles_only_date = available_only_date_df['TITLE'].unique()
    print(f"Found {len(unique_titles_only_date)} unique books with only publication date known.")

    # Sample titles
    if len(unique_titles_only_date) < sample_size_per_group:
        print(f"Warning: Only {len(unique_titles_only_date)} books available, sampling all of them.")
        actual_sample_size_date = len(unique_titles_only_date)
    else:
        actual_sample_size_date = sample_size_per_group
        
    sampled_titles_date = pd.Series(unique_titles_only_date).sample(
        n=actual_sample_size_date, replace=False, random_state=random_seed
    ).tolist()

    # Create the sample dataframe and then shuffle it to break any groupby sorting
    final_sample_only_date_df = available_only_date_df[available_only_date_df['TITLE'].isin(sampled_titles_date)]
    grouped_date_sample = final_sample_only_date_df.groupby('TITLE', group_keys=False).sample(n=1, random_state=random_seed)
    final_sample_only_date = grouped_date_sample.sample(frac=1, random_state=random_seed).reset_index(drop=True)
    print(f"Sampled {len(final_sample_only_date)} books.")

    # Add the newly sampled titles to the exclusion list for the next step
    all_excluded_titles.update(final_sample_only_date['TITLE'].unique())

    # --- Step 3: Sample books with ONLY author language known ---
    print("\n--- Sampling books with only author language ---")
    
    # Filter for rows where only language is known
    only_lang_df = df[~lang_unknown_mask & pub_date_unknown_mask]
    
    # Exclude all previously sampled books
    available_only_lang_df = only_lang_df[~only_lang_df['TITLE'].isin(all_excluded_titles)]
    unique_titles_only_lang = available_only_lang_df['TITLE'].unique()
    print(f"Found {len(unique_titles_only_lang)} unique books with only author language known.")

    # Sample titles
    if len(unique_titles_only_lang) < sample_size_per_group:
        print(f"Warning: Only {len(unique_titles_only_lang)} books available, sampling all of them.")
        actual_sample_size_lang = len(unique_titles_only_lang)
    else:
        actual_sample_size_lang = sample_size_per_group

    sampled_titles_lang = pd.Series(unique_titles_only_lang).sample(
        n=actual_sample_size_lang, replace=False, random_state=random_seed
    ).tolist()

    # Create the sample dataframe and then shuffle it
    final_sample_only_lang_df = available_only_lang_df[available_only_lang_df['TITLE'].isin(sampled_titles_lang)]
    grouped_lang_sample = final_sample_only_lang_df.groupby('TITLE', group_keys=False).sample(n=1, random_state=random_seed)
    final_sample_only_lang = grouped_lang_sample.sample(frac=1, random_state=random_seed).reset_index(drop=True)
    print(f"Sampled {len(final_sample_only_lang)} books.")

    # --- Step 4: Combine, sort, and save the final dataset ---
    print("\n--- Combining and saving samples ---")
    
    # Concatenate the two dataframes. Books with unknown language will appear first.
    final_combined_sample = pd.concat([final_sample_only_date, final_sample_only_lang], ignore_index=True)
    
    # The shuffle operation on the combined frame has been removed to preserve the order, 
    # ensuring that books with missing language data appear at the top of the output file.

    # Save to the new CSV file
    final_combined_sample.to_csv(output_path_partial, index=False)
    
    print(f"\nSuccessfully created a new combined sample of {len(final_combined_sample)} unique books.")
    print(f"File saved to: '{output_path_partial}'")

except FileNotFoundError as e:
    print(f"Error: A required file was not found. Details: {e}")
except NameError as e:
    print(f"Error: A required variable is not defined. Make sure you have run the initial cells. Details: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Loaded 9047 unique book titles from previous samples to exclude.

--- Sampling books with only publication date ---
Found 209 unique books with only publication date known.
Sampled 209 books.

--- Sampling books with only author language ---
Found 10843 unique books with only author language known.
Sampled 1000 books.

--- Combining and saving samples ---

Successfully created a new combined sample of 1209 unique books.
File saved to: 'random_book_sample_partial_data.csv'
